# The Op as an assembly

> A Gate is a decision point in an Op where the results of one or more
> Guards determine what happens next. Guards check. Gates decide.
>
> — [@oliphant2026, §5.2]

The section 5.6 manifest declares four such decisions:

> gates:
>   - if: confidence < 0.80
>     then: human_review_required
>   - if: consensus_disagreement > 0.25
>     then: expert_review_required
>   - if: sensitive_data_detected
>     then: stop_and_escalate
>   - if: vendor_risk == high
>     then: human_approval_required
>
> — [@oliphant2026, §5.6]

The model renders the Op as an assembly of abstract and concrete parts
wired along ports: four oracle components feed a policy engine across
reading seams; raised obligations cross to the human actions; clearance
crosses to the anomaly-summary cog, whose output the assembly's boundary
exposes. One toolchain then checks three levels at once:

- **component**: each `if/then` line as an `implies` constraint on the
  policy engine (the verbatim gates, with the numerical parameters
  factored out as single-point definitions);
- **wiring**: seam integrity (the value the engine ruled on is the value
  the oracle returned), plus structural seam conformance checked over
  the RDF conversion;
- **system**: properties only the composed Op has. Every raised
  obligation is discharged across components; a stopped Op emits
  nothing; and the aggregate outcome (executed vs noOp) must cohere
  with what was emitted. Navigating the rules and executing are
  different facts, and the model says which happened.

The internal wiring topology is not drawn in the paper; making it
explicit and checkable is a contribution of the computational form, and
is logged as such rather than passed off as the paper's own (GAP-08 in
the appendix).

## The factored parameters

The manifest's numbers appear in the model exactly once each, as named
parameters. A whitepaper can repeat a threshold; an implementation can
guarantee a single point of definition.

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.show_policy_parameters()

part def ValidationStrategyParameters {
            doc /* The numerical parameters of the policy, factored out as
               single-point definitions the gates reference — what an
               implementation can do that a whitepaper cannot. The defaults
               are the manifest's literals, verbatim; changing a threshold
               is a change HERE, nowhere else, and the Track records the
               values in force per run (WP section 5.3: Track contents
               include "the model versions and configuration parameters"). */
            attribute confidenceThreshold : ScalarValues::Real = 0.80;
            attribute consensusDisagreementThreshold : ScalarValues::Real = 0.25;
            attribute vendorRiskTrigger : RiskLevel = RiskLevel::high;
        }


## A gate as a constraint

Each gate keeps the manifest line in its documentation and states the
same rule as a machine-evaluable implication. One example:

In [2]:
exhibits.show_gate_checks()

requirement def <'GATE-01'> ConfidenceGate {
            doc /* - if: confidence < 0.80
                    then: human_review_required
               Source: WP section 5.6 gates block, verbatim. */
            subject pe : PolicyEngine;
            require constraint { pe.confidence < pe.policy.confidenceThreshold implies pe.humanReviewRequired }
        }


## The seams

Each dataflow between components is one declared seam, connected port
to port inside the assembly:

In [3]:
exhibits.show_seams()

interface confidenceSeam : ReadingSeam connect confidenceOracle.reading to policyEngine.fromConfidence;
interface consensusSeam : ReadingSeam connect consensusOracle.reading to policyEngine.fromConsensus;
interface scannerSeam : ReadingSeam connect scannerOracle.reading to policyEngine.fromScanner;
interface riskSeam : ReadingSeam connect riskOracle.reading to policyEngine.fromRisk;
interface obligationSeam : ObligationSeam connect policyEngine.toHuman to humanAction.fromEngine;
interface clearanceSeam : ClearanceSeam connect policyEngine.toCog to summaryCog.clearance;
interface w : StopWire connect driver.fire to op.stopSignal;
interface w : ProceedWire connect driver.fire to op.proceedSignal;


## The whole specification evaluates

Strict validation, then the satisfy sweep: every component, wiring, and
system check evaluated together against the committed run
configurations. The exit code is the finding.

In [4]:
exhibits.validate_and_satisfy()

validate -strict exit code: 0
✓ package VendorFraudReview
✓ satisfy cleanInterface01 holds
✓ satisfy cleanGate01 holds
✓ satisfy cleanGate02 holds
✓ satisfy cleanGate03 holds
✓ satisfy cleanGate04 holds
✓ satisfy cleanWire01 holds
✓ satisfy cleanWire02 holds
✓ satisfy cleanWire03 holds
✓ satisfy cleanWire04 holds
✓ satisfy cleanSystem01 holds
✓ satisfy cleanSystem02 holds
✓ satisfy cleanSystem03 holds
✓ satisfy escalatedInterface01 holds
✓ satisfy escalatedGate01 holds
✓ satisfy escalatedGate02 holds
✓ satisfy escalatedGate03 holds
✓ satisfy escalatedGate04 holds
✓ satisfy escalatedWire01 holds
✓ satisfy escalatedWire02 holds
✓ satisfy escalatedWire03 holds
✓ satisfy escalatedWire04 holds
✓ satisfy escalatedSystem01 holds
✓ satisfy escalatedSystem02 holds
✓ satisfy escalatedSystem03 holds
satisfy exit code: 0


## The specification can say no

A specification that cannot fail is a description. The counterexample
file holds three runs that must be refused, one per check level: a gate
fired but no obligation was raised (component); the obligation was
raised and never discharged (system); the Op stopped but transmitted
anyway (aggregate). The same evaluation names each violated
implication:

In [5]:
exhibits.refuse_counterexample()

✓ package VendorFraudReviewUnattended
✗ satisfy unattendedGate01 fails
  Required condition evaluated to false: pe.confidence < pe.policy.confidenceThreshold implies pe.humanReviewRequired
✗ satisfy undischargedSystem01 fails
  Required condition evaluated to false: op.policyEngine.humanReviewRequired implies op.humanAction.humanReviewPerformed
✗ satisfy emittedSystem02 fails
  Required condition evaluated to false: op.policyEngine.stopAndEscalate implies not op.summaryCog.outputEmitted
satisfy exit code: 1


## The seams themselves are checked

Strict validation accepts a seam whose ends do not match its interface
definition (probed against the pinned binary), so the structural wiring
rules live in code over the RDF conversion: the model holds the wiring,
the rules hold the wiring discipline. The same rule that passes the
committed assembly refuses a miswired counterexample in which an
oracle's reading port is wired into a summary input:

In [6]:
exhibits.check_wiring()

committed assembly (6 seams): wiring rules: OK
miswired counterexample: wiring rules: REFUSED
  confidenceSeam: end 'summaryCog.summaryIn' is urn:sysmlv2:element:VendorFraudReviewMiswired__SummaryWrite but the seam declares urn:sysmlv2:element:VendorFraudReviewMiswired__ReadingWrite


## The stop is a state

> If a Privacy Guard detects sensitive information, the Op stops before
> external transmission.
>
> — [@oliphant2026, §5.2]

The assembly exhibits a lifecycle (running to stopped, or running to
completed) that runs under the pinned executor: a driver component
delivers a signal across a connected seam to the assembly's own port,
and the trace shows the transition. The stop the paper describes is an
actual state the Op enters, not a label on a diagram:

In [7]:
exhibits.show_lifecycle()

stoppedContext
  [trace] enter: firing (entry action)
  [trace] enter: running
  [trace] enter: stopped
  [trace] transition: running -> stopped (event: accept Escalation)
  [trace] enter: spent
  [trace] transition: firing -> spent

completedContext
  [trace] enter: firing (entry action)
  [trace] enter: running
  [trace] enter: completed
  [trace] transition: running -> completed (event: accept Proceed)
  [trace] enter: spent
  [trace] transition: firing -> spent



## Conversion to RDF

The model converts to Turtle [@rdf2014] deterministically, and the
checks survive the conversion, so the same specification is queryable
alongside the vocabulary and the Track:

In [8]:
exhibits.check_conversion()

converted triples : 9768
byte-stable       : True
checks present    : ['GATE-01', 'GATE-02', 'GATE-03', 'GATE-04', 'WIRE-01', 'SYSTEM-01', 'SYSTEM-02', 'SYSTEM-03']
